# Dataset download, metadata checks and speaker selection

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import sys
import os
import json
import platform
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata

import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

folders = [
    "data/metadata",
    "data/processed",
    "features/acoustic",
    "features/linguistic",
    "models",
    "results/tables",
    "results/figures",
    "reports",
    "checkpoints",
    "paper",
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

packages = [
    "numpy", "pandas", "scipy", "scikit-learn",
    "librosa", "soundfile", "torch",
    "transformers", "jiwer", "parselmouth"
]

versions = {}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ] if torch.cuda.is_available() else [],
    "packages": versions,
}

version_file = PROJECT / "reports/environment_versions.txt"
version_file.write_text(
    json.dumps(info, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress_file = PROJECT / "checkpoints/research_progress.json"

if not progress_file.exists():
    progress = {
        "project": "ASR Fluency Robustness",
        "current_step": 1,
        "completed_steps": [0],
        "dataset": "Speechocean762",
        "total_audio_files": None,
        "processed_audio_files": 0,
        "failed_audio_files": 0,
        "random_seed": 42,
        "next_task": "Verify Kaggle dataset access",
        "updated_at": datetime.now(timezone.utc).isoformat()
    }
    progress_file.write_text(
        json.dumps(progress, indent=2),
        encoding="utf-8"
    )

print("=== ENVIRONMENT ===")
print(json.dumps(info, indent=2))

print("\n=== INPUT DATASETS ===")
input_dir = Path("/kaggle/input")

if input_dir.exists():
    print([p.name for p in input_dir.iterdir()])
else:
    print("Kaggle input directory not found")

print("\n=== PROJECT ===")
print(PROJECT)
print("Environment and checkpoint files created.")


In [ ]:

import urllib.request
import json

url = (
    "https://huggingface.co/api/datasets/"
    "mispeech/speechocean762"
)

try:
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "research-dataset-check/1.0"}
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        data = json.load(response)

    print("=== DATASET CONNECTION ===")
    print("Status: SUCCESS")
    print("Dataset:", data.get("id"))
    print("Private:", data.get("private"))
    print("Revision:", data.get("sha"))

    info = data.get("cardData") or {}
    print("License:", info.get("license"))

    print("\n=== DATASET FILES ===")
    for item in data.get("siblings", []):
        filename = item.get("rfilename", "")
        if filename.endswith((".parquet", ".json", ".md")):
            print(filename)

except Exception as e:
    print("Status: FAILED")
    print("Error:", type(e).__name__, str(e))


In [ ]:

import io
import urllib.request
import pyarrow.parquet as pq

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

url = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

print("=== PARQUET SCHEMA CHECK ===")

try:
    request = urllib.request.Request(
        url,
        headers={"Range": "bytes=0-65535"}
    )

    with urllib.request.urlopen(request, timeout=60) as response:
        data = response.read()
        status = response.status

    print("HTTP status:", status)
    print("Bytes downloaded:", len(data))

    if status == 206:
        print("Partial download supported.")
        print("First bytes:", data[:4])
        print(
            "Full Parquet schema requires the footer. "
            "We will use a different metadata method."
        )
    else:
        print("Server returned full file.")
        pf = pq.ParquetFile(io.BytesIO(data))

        print("Total rows:", pf.metadata.num_rows)
        print("Row groups:", pf.metadata.num_row_groups)
        print("\n=== COLUMNS ===")
        print(pf.schema_arrow)

except Exception as e:
    print("Error:", type(e).__name__, str(e))


In [ ]:

import pyarrow.parquet as pq
import pyarrow.fs as pafs

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

url = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

print("=== REMOTE PARQUET METADATA ===")

try:
    filesystem, path = pafs.FileSystem.from_uri(url)

    parquet_file = pq.ParquetFile(
        path,
        filesystem=filesystem
    )

    print("Total rows:", parquet_file.metadata.num_rows)
    print("Row groups:", parquet_file.metadata.num_row_groups)

    print("\n=== COLUMNS ===")
    print(parquet_file.schema_arrow)

except Exception as e:
    print("FAILED:", type(e).__name__)
    print(str(e))


In [ ]:

import urllib.request
import struct
import pyarrow.parquet as pq
import pyarrow as pa

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

def read_range(start, end):
    req = urllib.request.Request(
        URL,
        headers={
            "Range": f"bytes={start}-{end}",
            "Accept-Encoding": "identity"
        }
    )

    with urllib.request.urlopen(req, timeout=60) as response:
        if response.status != 206:
            raise RuntimeError(
                f"Expected HTTP 206, got {response.status}"
            )
        return response.read()

print("=== PARQUET FOOTER CHECK ===")

# 1. Get the last 8 bytes and total file size
req = urllib.request.Request(
    URL,
    headers={
        "Range": "bytes=-8",
        "Accept-Encoding": "identity"
    }
)

with urllib.request.urlopen(req, timeout=60) as response:
    if response.status != 206:
        raise RuntimeError("HTTP Range not supported")

    footer = response.read()
    content_range = response.headers["Content-Range"]

file_size = int(content_range.split("/")[-1])

print("File size:", file_size)
print("Footer bytes:", footer)

if len(footer) != 8 or footer[-4:] != b"PAR1":
    raise ValueError("Invalid Parquet footer")

metadata_length = struct.unpack("<I", footer[:4])[0]

print("Metadata length:", metadata_length)

# 2. Fetch metadata bytes
metadata_start = file_size - 8 - metadata_length

metadata_bytes = read_range(
    metadata_start,
    file_size - 9
)

# 3. Construct a minimal Parquet file
minimal_parquet = (
    b"PAR1"
    + metadata_bytes
    + footer
)

# Metadata can contain row-group offsets referring
# to the original file. Read only schema/metadata.
try:
    reader = pq.ParquetFile(
        pa.BufferReader(minimal_parquet)
    )

    print("\n=== DATASET METADATA ===")
    print("Total rows:", reader.metadata.num_rows)
    print("Row groups:", reader.metadata.num_row_groups)

    print("\n=== COLUMNS ===")
    print(reader.schema_arrow)

except Exception as e:
    print("\nMetadata extraction requires adjustment:")
    print(type(e).__name__, str(e))


In [ ]:

from pathlib import Path
import requests
import time
import os
import json

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

EXPECTED_SIZE = 332843486

if TARGET.exists() and TARGET.stat().st_size == EXPECTED_SIZE:
    print("Train file already downloaded.")
else:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = {}
        if existing:
            headers["Range"] = f"bytes={existing}-"

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                if existing and response.status == 200:
                    # Server ignored the Range header
                    existing = 0

                mode = "ab" if existing and response.status == 206 else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Download completed.")
                break

            print("Incomplete file. Retrying...")

        except Exception as e:
            print(f"Attempt {attempt + 1}: {e}")
            time.sleep(3)

    else:
        raise RuntimeError("Download incomplete after retries.")

print("=== TRAIN DATASET ===")
print("File:", TARGET)
print("Size:", TARGET.stat().st_size)

# Verify Parquet integrity and row count
import pyarrow.parquet as pq

pf = pq.ParquetFile(TARGET)

print("Rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)
print("Columns:", pf.schema_arrow.names)

# Update checkpoint
progress_file = PROJECT / "checkpoints" / "research_progress.json"

progress = json.loads(progress_file.read_text(encoding="utf-8"))
progress["train_downloaded"] = True
progress["train_rows"] = pf.metadata.num_rows
progress["dataset_revision"] = REVISION
progress["next_task"] = "Inspect train metadata and speaker IDs"

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nCheckpoint updated.")


In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = (
    PROJECT / "data/raw/speechocean762_train.parquet"
)

# Read metadata only, without audio bytes
columns = [
    "speaker",
    "gender",
    "age",
    "fluency",
    "accuracy",
    "completeness",
    "prosodic",
    "total",
    "text"
]

df = pq.read_table(
    TRAIN,
    columns=columns
).to_pandas()

print("=== BASIC INFORMATION ===")
print("Total rows:", len(df))
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate rows:", df.duplicated().sum())

print("\n=== FLUENCY SCORES ===")
print(df["fluency"].describe())

print("\nScore distribution:")
print(df["fluency"].value_counts().sort_index())

print("\n=== SPEAKER DISTRIBUTION ===")
print(df.groupby("speaker").size().describe())

print("\n=== GENDER ===")
print(df["gender"].value_counts(dropna=False))

print("\n=== AGE ===")
print(df["age"].describe())

print("\n=== MISSING VALUES ===")
print(df.isna().sum())

print("\n=== SAMPLE RECORDS ===")
print(
    df[
        ["speaker", "fluency", "text"]
    ].head(5).to_string(index=False)
)

# Save compact initial metadata
output = PROJECT / "data/metadata/train_metadata_initial.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8"
)

print("\nMetadata saved:", output)


In [ ]:

import io
from pathlib import Path
import pyarrow.parquet as pq
import soundfile as sf
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

# Read only first row group and audio column
pf = pq.ParquetFile(TRAIN)

table = pf.read_row_group(
    0,
    columns=["audio"]
)

audio_values = table.column("audio").to_pylist()

print("=== AUDIO INSPECTION ===")
print("Row group audio count:", len(audio_values))

records = []

for i, audio in enumerate(audio_values[:5], start=1):

    audio_bytes = audio.get("bytes")
    audio_path = audio.get("path")

    if not audio_bytes:
        print(f"Audio {i}: missing bytes")
        continue

    with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
        sample_rate = f.samplerate
        channels = f.channels
        frames = f.frames
        duration = frames / sample_rate
        audio_format = f.format
        subtype = f.subtype

    records.append({
        "row": i,
        "path": audio_path,
        "sample_rate": sample_rate,
        "channels": channels,
        "duration_sec": round(duration, 3),
        "format": audio_format,
        "subtype": subtype
    })

result = pd.DataFrame(records)

print(result.to_string(index=False))

output = PROJECT / "data/metadata/audio_inspection_5.csv"
result.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

OUT = PROJECT / "data/metadata/train_audio_technical_metadata.csv"
ERRORS = PROJECT / "data/metadata/train_audio_errors.csv"
PROGRESS = PROJECT / "checkpoints/research_progress.json"

pf = pq.ParquetFile(TRAIN)

records = []
errors = []
global_row = 0

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(group_id, columns=["audio"])
    audios = table.column("audio").to_pylist()

    for audio in audios:
        global_row += 1

        try:
            audio_bytes = audio.get("bytes")
            audio_path = audio.get("path")

            if not audio_bytes:
                raise ValueError("Missing audio bytes")

            with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
                records.append({
                    "row": global_row,
                    "audio_id": Path(audio_path).stem,
                    "audio_path": audio_path,
                    "sample_rate": f.samplerate,
                    "channels": f.channels,
                    "duration_sec": round(f.frames / f.samplerate, 4),
                    "format": f.format,
                    "subtype": f.subtype
                })

        except Exception as e:
            errors.append({
                "row": global_row,
                "error": str(e)
            })

    print(
        f"Row group {group_id + 1}/"
        f"{pf.metadata.num_row_groups} completed."
    )

    # Checkpoint after each row group
    pd.DataFrame(records).to_csv(OUT, index=False)
    pd.DataFrame(errors, columns=["row", "error"]).to_csv(
        ERRORS, index=False
    )

df_audio = pd.DataFrame(records)

print("\n=== FULL AUDIO VALIDATION ===")
print("Total examined:", global_row)
print("Successful:", len(records))
print("Failed:", len(errors))

print("\nSample rate distribution:")
print(df_audio["sample_rate"].value_counts())

print("\nChannel distribution:")
print(df_audio["channels"].value_counts())

print("\nFormat distribution:")
print(df_audio["format"].value_counts())

print("\nDuration statistics:")
print(df_audio["duration_sec"].describe())

print("\nUnique audio IDs:", df_audio["audio_id"].nunique())
print("Duplicate audio IDs:", df_audio["audio_id"].duplicated().sum())

progress = json.loads(PROGRESS.read_text(encoding="utf-8"))

progress["audio_technical_validation"] = {
    "examined": global_row,
    "successful": len(records),
    "failed": len(errors),
    "metadata_file": str(OUT),
    "error_file": str(ERRORS)
}
progress["next_task"] = "Validate metadata joins and train/test strategy"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

PROGRESS.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nFiles saved:")
print(OUT)
print(ERRORS)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

META = PROJECT / "data/metadata/train_metadata_initial.csv"
AUDIO = PROJECT / "data/metadata/train_audio_technical_metadata.csv"
OUTPUT = PROJECT / "data/metadata/dataset_metadata_train.csv"
PROGRESS = PROJECT / "checkpoints/research_progress.json"

df_meta = pd.read_csv(
    META,
    dtype={"speaker": "string"}
)

df_audio = pd.read_csv(
    AUDIO,
    dtype={"audio_id": "string", "audio_path": "string"}
)

# Original Parquet row order is preserved in both CSVs.
df_meta.insert(0, "row", range(1, len(df_meta) + 1))

assert len(df_meta) == 2500
assert len(df_audio) == 2500
assert df_meta["row"].is_unique
assert df_audio["row"].is_unique
assert df_audio["audio_id"].is_unique
assert df_audio["row"].between(1, len(df_meta)).all()

df = df_meta.merge(
    df_audio,
    on="row",
    how="inner",
    validate="one_to_one"
)

assert len(df) == 2500
assert df["fluency"].notna().all()
assert df["speaker"].notna().all()
assert df["audio_id"].notna().all()

df.insert(1, "split", "train")

# Check the audio ID against the speaker ID
# before relying on this relationship.
df["audio_speaker_prefix"] = df["audio_id"].str[:4]

prefix_matches = (
    df["audio_speaker_prefix"] == df["speaker"]
)

print("=== METADATA JOIN VALIDATION ===")
print("Total rows:", len(df))
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Missing fluency:", df["fluency"].isna().sum())
print("Speaker-prefix matches:", prefix_matches.sum())
print("Speaker-prefix mismatches:", (~prefix_matches).sum())

print("\n=== SAMPLE ===")
print(
    df[
        [
            "row", "audio_id", "speaker",
            "fluency", "duration_sec", "text"
        ]
    ].head(5).to_string(index=False)
)

# Keep speaker IDs as strings to preserve leading zeros.
df.to_csv(OUTPUT, index=False, encoding="utf-8")

progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
progress["train_metadata_file"] = str(OUTPUT)
progress["train_metadata_rows"] = len(df)
progress["next_task"] = "Validate train speaker identities and test split"
PROGRESS.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nSaved:", OUTPUT)
print("Checkpoint updated.")


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FILE = PROJECT / "data/metadata/dataset_metadata_train.csv"

df = pd.read_csv(
    FILE,
    dtype={"speaker": "string", "audio_id": "string"}
)

print("=== AUDIO ID STRUCTURE ===")
print("Audio ID lengths:")
print(df["audio_id"].str.len().value_counts().sort_index())

print("\nFirst 15 examples:")
print(
    df[["audio_id", "speaker"]]
    .head(15)
    .to_string(index=False)
)

print("\n=== POSITION MATCH ANALYSIS ===")

for start in range(6):
    extracted = df["audio_id"].str.slice(start, start + 4)
    matches = (extracted == df["speaker"]).sum()

    print(
        f"Positions {start}:{start+4} "
        f"=> matches: {matches}/2500"
    )

print("\n=== SPEAKER EXAMPLES ===")
sample_speakers = df["speaker"].drop_duplicates().head(5)

for speaker in sample_speakers:
    values = (
        df.loc[df["speaker"] == speaker, "audio_id"]
        .head(5)
        .tolist()
    )
    print(f"Speaker {speaker}: {values}")

print("\n=== BASIC VALIDATION ===")
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate audio IDs:", df["audio_id"].duplicated().sum())


In [ ]:

from pathlib import Path

print("=== KAGGLE FILE RECOVERY CHECK ===")

locations = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/kaggle/working/acoustic_linguistic_research"),
]

for folder in locations:
    print(f"\nDirectory: {folder}")

    if not folder.exists():
        print("NOT FOUND")
        continue

    files = list(folder.iterdir())

    if not files:
        print("EMPTY")
    else:
        for item in files[:30]:
            if item.is_file():
                print(
                    "FILE:",
                    item.name,
                    "| Size:",
                    round(item.stat().st_size / 1024 / 1024, 2),
                    "MB"
                )
            else:
                print("DIR:", item.name)

print("\n=== IMPORTANT FILES ===")

project = Path(
    "/kaggle/working/acoustic_linguistic_research"
)

important_files = [
    "data/raw/speechocean762_train.parquet",
    "data/metadata/train_metadata_initial.csv",
    "data/metadata/train_audio_technical_metadata.csv",
    "data/metadata/dataset_metadata_train.csv",
    "checkpoints/research_progress.json",
]

for filename in important_files:
    path = project / filename
    print(
        "FOUND" if path.exists() else "MISSING",
        filename
    )


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import io
import json
import os
import zipfile

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
META = PROJECT / "data/metadata"
CHECK = PROJECT / "checkpoints"

for folder in [RAW, META, CHECK]:
    folder.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"
EXPECTED_SIZE = 332843486

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

# Download with resume support
if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0
        headers = {"Range": f"bytes={existing}-"} if existing else {}

        try:
            with requests.get(
                URL, headers=headers, stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                resume = existing > 0 and response.status == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                break

        except Exception as e:
            print("Retry:", attempt + 1, str(e))

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Train dataset download failed")

print("Train Parquet restored.")

pf = pq.ParquetFile(TARGET)
assert pf.metadata.num_rows == 2500

# Read non-audio metadata
columns = [
    "speaker", "gender", "age", "fluency",
    "accuracy", "completeness", "prosodic",
    "total", "text"
]

df_meta = pq.read_table(
    TARGET, columns=columns
).to_pandas()

df_meta.insert(0, "row", range(1, len(df_meta) + 1))
df_meta["speaker"] = df_meta["speaker"].astype("string")

# Restore audio metadata
records = []
row_number = 0

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        row_number += 1
        audio_bytes = audio["bytes"]
        audio_path = audio["path"]

        if not audio_bytes:
            raise ValueError(f"Missing audio at row {row_number}")

        with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
            records.append({
                "row": row_number,
                "audio_id": Path(audio_path).stem,
                "audio_path": audio_path,
                "sample_rate": f.samplerate,
                "channels": f.channels,
                "duration_sec": round(f.frames / f.samplerate, 4),
                "format": f.format,
                "subtype": f.subtype
            })

    print(f"Audio metadata: group {group_id + 1}/{pf.metadata.num_row_groups}")

df_audio = pd.DataFrame(records)

df = df_meta.merge(
    df_audio, on="row",
    how="inner", validate="one_to_one"
)
df.insert(1, "split", "train")

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125
assert df["fluency"].notna().all()
assert (df["sample_rate"] == 16000).all()
assert (df["channels"] == 1).all()

df_meta.to_csv(META / "train_metadata_initial.csv", index=False)
df_audio.to_csv(META / "train_audio_technical_metadata.csv", index=False)
df.to_csv(META / "dataset_metadata_train.csv", index=False)

progress = {
    "project": "ASR Fluency Robustness",
    "current_step": 1,
    "completed_steps": [0],
    "dataset": "Speechocean762",
    "dataset_revision": REVISION,
    "train_rows": len(df),
    "train_speakers": int(df["speaker"].nunique()),
    "audio_validation_successful": len(df_audio),
    "audio_validation_failed": 0,
    "random_seed": 42,
    "next_task": "Investigate audio ID and speaker mapping",
    "updated_at": datetime.now(timezone.utc).isoformat()
}

(CHECK / "research_progress.json").write_text(
    json.dumps(progress, indent=2),
    encoding="utf-8"
)

# Backup metadata and checkpoint (not audio)
ZIP = Path("/kaggle/working/asr_fluency_step1_backup.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [META, CHECK]:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(PROJECT))

print("\n=== RECOVERY COMPLETE ===")
print("Train rows:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Unique audio IDs:", df["audio_id"].nunique())
print("Metadata restored:", (META / "dataset_metadata_train.csv").exists())
print("Checkpoint restored:", (CHECK / "research_progress.json").exists())
print("Backup ZIP:", ZIP)
print("Backup size (MB):", round(ZIP.stat().st_size / 1048576, 2))


In [ ]:

import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FILE = PROJECT / "data/metadata/dataset_metadata_train.csv"

df = pd.read_csv(
    FILE,
    dtype={"speaker": "string", "audio_id": "string"}
)

print("=== AUDIO ID STRUCTURE ===")
print("Audio ID lengths:")
print(df["audio_id"].str.len().value_counts().sort_index())

print("\nFirst 15 examples:")
print(
    df[["audio_id", "speaker"]]
    .head(15)
    .to_string(index=False)
)

print("\n=== POSITION MATCH ANALYSIS ===")

for start in range(6):
    extracted = df["audio_id"].str.slice(start, start + 4)
    matches = (extracted == df["speaker"]).sum()

    print(
        f"Positions {start}:{start + 4} "
        f"=> matches: {matches}/{len(df)}"
    )

print("\n=== SPEAKER EXAMPLES ===")

for speaker in df["speaker"].drop_duplicates().head(5):
    audio_ids = df.loc[
        df["speaker"] == speaker, "audio_id"
    ].head(5).tolist()

    print(f"Speaker {speaker}: {audio_ids}")

print("\n=== VALIDATION ===")
print("Rows:", len(df))
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate audio IDs:", df["audio_id"].duplicated().sum())

# Save the actual inspection results
report = PROJECT / "reports/audio_speaker_mapping_check.txt"
report.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "STEP 1 - Audio Speaker Mapping Check",
    f"Rows: {len(df)}",
    f"Unique audio IDs: {df['audio_id'].nunique()}",
    f"Unique speakers: {df['speaker'].nunique()}",
    "",
    "Position matches:"
]

for start in range(6):
    matches = (
        df["audio_id"].str.slice(start, start + 4)
        == df["speaker"]
    ).sum()
    lines.append(f"{start}:{start + 4} = {matches}")

report.write_text("\n".join(lines), encoding="utf-8")

progress_path = PROJECT / "checkpoints/research_progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8"))
progress["speaker_mapping_check"] = "completed"
progress["next_task"] = "Verify official test speaker independence"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

progress_path.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nReport saved:", report)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import requests
import pyarrow.parquet as pq
import pandas as pd
import json
import os
import time

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/test-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_test.parquet"
PART = RAW / "speechocean762_test.parquet.part"

# Get expected file size using a 1-byte Range request
response = requests.get(
    URL,
    headers={"Range": "bytes=0-0"},
    timeout=60
)
response.raise_for_status()

if response.status_code != 206:
    raise RuntimeError("HTTP Range response required")

EXPECTED_SIZE = int(
    response.headers["Content-Range"].split("/")[-1]
)

print("Expected test file size:", EXPECTED_SIZE)

# Download with retry and resume
if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:

    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = (
            {"Range": f"bytes={existing}-"}
            if existing else {}
        )

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as r:
                r.raise_for_status()

                resume = existing > 0 and r.status_code == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Test file downloaded.")
                break

            print("Incomplete download, retrying...")

        except Exception as e:
            print("Attempt:", attempt + 1, str(e))
            time.sleep(3)

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Test download failed")

# Read speaker column only
test_speakers = pq.read_table(
    TARGET,
    columns=["speaker"]
).to_pandas()["speaker"].astype("string")

train_meta = pd.read_csv(
    PROJECT / "data/metadata/dataset_metadata_train.csv",
    dtype={"speaker": "string"}
)

train_ids = set(train_meta["speaker"].dropna())
test_ids = set(test_speakers.dropna())

overlap = train_ids.intersection(test_ids)

print("\n=== SPEAKER INDEPENDENCE ===")
print("Train speakers:", len(train_ids))
print("Test speakers:", len(test_ids))
print("Shared speakers:", len(overlap))
print("Test rows:", len(test_speakers))
print("Missing test speaker IDs:", test_speakers.isna().sum())

if overlap:
    print("WARNING - overlapping speakers:", sorted(overlap))
else:
    print("PASS - official splits are speaker-disjoint")

assert len(test_speakers) == 2500
assert test_speakers.notna().all()
assert len(overlap) == 0, "Speaker leakage detected!"

# Save verification report
REPORT = PROJECT / "reports/speaker_independence_report.json"
REPORT.parent.mkdir(parents=True, exist_ok=True)

report = {
    "dataset": "Speechocean762",
    "revision": REVISION,
    "train_speakers": len(train_ids),
    "test_speakers": len(test_ids),
    "shared_speakers": sorted(overlap),
    "test_rows": len(test_speakers),
    "speaker_independent": len(overlap) == 0
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# Update checkpoint
PROGRESS = PROJECT / "checkpoints/research_progress.json"
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
progress["test_downloaded"] = True
progress["speaker_independence_verified"] = len(overlap) == 0
progress["next_task"] = "Finalize STEP 1 dataset metadata"
PROGRESS.write_text(
    json.dumps(progress, indent=2),
    encoding="utf-8"
)

print("\nReport saved:", REPORT)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import pyarrow.parquet as pq
import importlib.metadata as metadata
import platform
import json
import zipfile
import hashlib
import sys

from IPython.display import display, FileLink

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
TEST = PROJECT / "data/raw/speechocean762_test.parquet"
META = PROJECT / "data/metadata/dataset_metadata_train.csv"
REPORT_DIR = PROJECT / "reports"
CHECK_DIR = PROJECT / "checkpoints"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECK_DIR.mkdir(parents=True, exist_ok=True)

# Verify files before finalizing STEP 1
assert TRAIN.exists() and TEST.exists() and META.exists()

train_pf = pq.ParquetFile(TRAIN)
test_pf = pq.ParquetFile(TEST)

assert train_pf.metadata.num_rows == 2500
assert test_pf.metadata.num_rows == 2500

df = pd.read_csv(
    META,
    dtype={"speaker": "string", "audio_id": "string"}
)

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125
assert df["fluency"].notna().all()

test_speakers = set(
    pq.read_table(TEST, columns=["speaker"])
    .to_pandas()["speaker"].astype(str)
)
train_speakers = set(df["speaker"].astype(str))

assert len(train_speakers & test_speakers) == 0

# Record installed package versions
packages = [
    "numpy", "pandas", "pyarrow", "scipy",
    "scikit-learn", "librosa", "soundfile",
    "torch", "transformers", "requests",
    "jiwer", "praat-parselmouth"
]

versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {}
}

for package in packages:
    try:
        versions["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions["packages"][package] = "NOT INSTALLED"

(REPORT_DIR / "environment_versions.txt").write_text(
    json.dumps(versions, indent=2),
    encoding="utf-8"
)

report_text = f"""# STEP 1 — Environment and Dataset Verification

Date (UTC): {datetime.now(timezone.utc).isoformat()}

Dataset: Speechocean762
Dataset revision: 06385584fad212b26134c656fdd3ccf9f093f33e

## Verified results

- Train utterances: 2500
- Test utterances: 2500
- Train speakers: 125
- Test speakers: {len(test_speakers)}
- Train/test shared speakers: 0
- Train unique audio IDs: 2500
- Train missing fluency values: 0
- Train audio technical validation: 2500/2500 successful
- Train audio sample rate: 16000 Hz
- Train audio channels: mono
- Train audio format: WAV
- Train audio mean duration: 4.141644 seconds
- Train fluency mean: 7.7284
- Train fluency SD: 1.555249
- Speaker mapping: audio_id[1:5] matches speaker for all train rows

## Research safeguards

- Official test labels have not been inspected.
- Final test split is reserved for independent evaluation.
- Internal validation must be speaker-independent.
- No ASR transcription or ML training has been performed.

## Next step

STEP 2: Pilot acoustic feature extraction.
Before modeling, prepare train-only speaker-independent
validation and preserve all output checkpoints.

## Important

Kaggle working directory is temporary.
Restore metadata from the ZIP backup in future sessions.
Raw Parquet files must be reattached or downloaded separately.
"""

(REPORT_DIR / "step01_report.md").write_text(
    report_text, encoding="utf-8"
)

progress_path = CHECK_DIR / "research_progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8"))

progress.update({
    "current_step": 1,
    "completed_steps": [0, 1],
    "dataset_revision":
        "06385584fad212b26134c656fdd3ccf9f093f33e",
    "train_rows": 2500,
    "test_rows": 2500,
    "train_speakers": 125,
    "test_speakers": len(test_speakers),
    "shared_speakers": 0,
    "speaker_independence_verified": True,
    "next_step": "STEP 2 - Pilot acoustic feature extraction",
    "updated_at": datetime.now(timezone.utc).isoformat()
})

progress_path.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ZIP all metadata, reports and checkpoints.
# Exclude large raw audio/Parquet files.
ZIP = Path("/kaggle/working/asr_fluency_step01_complete.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [
        PROJECT / "data/metadata",
        REPORT_DIR,
        CHECK_DIR
    ]:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(PROJECT))

# Confirm ZIP integrity
with zipfile.ZipFile(ZIP) as z:
    bad_file = z.testzip()
    assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

sha256 = hashlib.sha256()
with open(ZIP, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(block)

print("=== STEP 1 COMPLETED ===")
print("Train rows:", train_pf.metadata.num_rows)
print("Test rows:", test_pf.metadata.num_rows)
print("Train speakers:", len(train_speakers))
print("Test speakers:", len(test_speakers))
print("Shared speakers:", len(train_speakers & test_speakers))
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha256.hexdigest())
print("ZIP integrity: PASS")

display(FileLink(str(ZIP)))


In [ ]:

from pathlib import Path
import zipfile
import json
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PROJECT.mkdir(parents=True, exist_ok=True)

EXPECTED_ZIP = "asr_fluency_step01_complete.zip"

# Find the uploaded backup in Kaggle input or working
candidates = [
    p for root in [
        Path("/kaggle/input"),
        Path("/kaggle/working")
    ]
    if root.exists()
    for p in root.rglob(EXPECTED_ZIP)
]

print("=== STEP 2 BACKUP RECOVERY ===")

if not candidates:
    print("BACKUP NOT FOUND")
    print("Upload the STEP 1 ZIP via Kaggle Add Input.")
else:
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one backup, found {len(candidates)}: "
            f"{candidates}"
        )

    backup = candidates[0]
    print("Backup found:", backup)

    with zipfile.ZipFile(backup) as z:
        assert z.testzip() is None, "ZIP integrity failed"

        # Refuse unsafe paths
        for item in z.infolist():
            target = (PROJECT / item.filename).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise RuntimeError(
                    f"Unsafe ZIP entry: {item.filename}"
                )

        z.extractall(PROJECT)

    metadata_file = (
        PROJECT / "data/metadata/dataset_metadata_train.csv"
    )
    progress_file = (
        PROJECT / "checkpoints/research_progress.json"
    )

    assert metadata_file.exists()
    assert progress_file.exists()

    df = pd.read_csv(
        metadata_file,
        dtype={
            "speaker": "string",
            "audio_id": "string"
        }
    )

    progress = json.loads(
        progress_file.read_text(encoding="utf-8")
    )

    assert len(df) == 2500
    assert df["speaker"].nunique() == 125
    assert df["audio_id"].is_unique
    assert df["fluency"].notna().all()
    assert progress["completed_steps"] == [0, 1]

    print("\n=== RECOVERED METADATA ===")
    print("Rows:", len(df))
    print("Speakers:", df["speaker"].nunique())
    print("Unique audio IDs:", df["audio_id"].nunique())
    print("Missing fluency:", df["fluency"].isna().sum())

    print("\n=== CHECKPOINT ===")
    print("Completed steps:", progress["completed_steps"])
    print("Dataset:", progress["dataset"])
    print("Revision:", progress["dataset_revision"])
    print("Random seed:", progress["random_seed"])

    print("\nRECOVERY SUCCESSFUL")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "data/metadata/dataset_metadata_train.csv"
OUTPUT = PROJECT / "data/metadata/pilot_selection_400.csv"
SPEAKERS_FILE = PROJECT / "checkpoints/pilot_speaker_ids.json"
PROGRESS_FILE = PROJECT / "checkpoints/research_progress.json"

SEED = 42
PILOT_SPEAKERS = 20
EXPECTED_AUDIO = 400

df = pd.read_csv(
    SOURCE,
    dtype={"speaker": "string", "audio_id": "string"}
)

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125

# Do not change an existing pilot selection
if OUTPUT.exists() and SPEAKERS_FILE.exists():

    pilot = pd.read_csv(
        OUTPUT,
        dtype={"speaker": "string", "audio_id": "string"}
    )

    saved_speakers = json.loads(
        SPEAKERS_FILE.read_text(encoding="utf-8")
    )

    assert set(pilot["speaker"]) == set(
        saved_speakers["speaker_ids"]
    )

    print("Existing pilot selection restored.")

else:
    rng = np.random.default_rng(SEED)

    speaker_ids = sorted(df["speaker"].unique())

    selected = sorted(
        rng.choice(
            speaker_ids,
            size=PILOT_SPEAKERS,
            replace=False
        ).tolist()
    )

    pilot = df[df["speaker"].isin(selected)].copy()

    pilot = pilot.sort_values(
        ["speaker", "audio_id"]
    ).reset_index(drop=True)

    pilot.insert(
        0, "pilot_index",
        range(1, len(pilot) + 1)
    )

    assert len(pilot) == EXPECTED_AUDIO
    assert pilot["speaker"].nunique() == PILOT_SPEAKERS
    assert pilot["audio_id"].is_unique

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    SPEAKERS_FILE.parent.mkdir(parents=True, exist_ok=True)

    pilot.to_csv(OUTPUT, index=False)

    SPEAKERS_FILE.write_text(
        json.dumps(
            {
                "seed": SEED,
                "speaker_ids": selected,
                "pilot_audio_count": len(pilot)
            },
            indent=2
        ),
        encoding="utf-8"
    )

print("=== PILOT SELECTION ===")
print("Pilot audio:", len(pilot))
print("Pilot speakers:", pilot["speaker"].nunique())
print("Audio per speaker:")
print(pilot.groupby("speaker").size().describe())

print("\n=== FLUENCY DISTRIBUTION ===")
print(pilot["fluency"].value_counts().sort_index())

print("\n=== PILOT SUMMARY ===")
print(pilot["fluency"].describe())

print("\n=== SELECTED SPEAKERS ===")
print(sorted(pilot["speaker"].unique().tolist()))

print("\n=== SAMPLE ===")
print(
    pilot[
        ["pilot_index", "audio_id", "speaker",
         "fluency", "duration_sec"]
    ].head(5).to_string(index=False)
)

progress = json.loads(
    PROGRESS_FILE.read_text(encoding="utf-8")
)

progress["current_step"] = 2
progress["pilot_audio_count"] = len(pilot)
progress["pilot_speaker_count"] = pilot["speaker"].nunique()
progress["pilot_seed"] = SEED
progress["pilot_selection_file"] = str(OUTPUT)
progress["next_task"] = "Restore train audio and begin pilot extraction"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

PROGRESS_FILE.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nPilot selection saved:", OUTPUT)
print("Speaker IDs saved:", SPEAKERS_FILE)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import requests
import pyarrow.parquet as pq
import pandas as pd
import json
import os
import time

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"
EXPECTED_SIZE = 332843486

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

if TARGET.exists() and TARGET.stat().st_size == EXPECTED_SIZE:
    print("Existing train Parquet found.")
else:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = (
            {"Range": f"bytes={existing}-"}
            if existing else {}
        )

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                resume = existing > 0 and response.status_code == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Train Parquet downloaded.")
                break

            print("Incomplete download, retrying...")

        except Exception as e:
            print(f"Attempt {attempt + 1}: {e}")
            time.sleep(3)

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Train Parquet recovery failed.")

pf = pq.ParquetFile(TARGET)

assert pf.metadata.num_rows == 2500

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

train_meta = pd.read_csv(
    PROJECT / "data/metadata/dataset_metadata_train.csv",
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert set(pilot["audio_id"]).issubset(
    set(train_meta["audio_id"])
)

print("\n=== STEP 2 AUDIO RECOVERY ===")
print("Train Parquet size:", TARGET.stat().st_size)
print("Train rows:", pf.metadata.num_rows)
print("Pilot audio:", len(pilot))
print("Pilot speakers:", pilot["speaker"].nunique())
print("Pilot IDs matched:", len(
    set(pilot["audio_id"]) & set(train_meta["audio_id"])
))

progress_file = PROJECT / "checkpoints/research_progress.json"

progress = json.loads(
    progress_file.read_text(encoding="utf-8")
)

progress["step2_train_audio_available"] = True
progress["step2_pilot_ids_verified"] = True
progress["next_task"] = "Pilot acoustic feature extraction setup"

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Checkpoint updated.")
